# Test Scores Analysis
This notebook analyzes test scores from `test_scores.csv`. The metrics in the CSV are stored as string tuples formatted like `(score, weight)`, so this notebook parses them to calculate accurate weighted averages.

In [15]:
import pandas as pd
import ast

# Path to the data (relative to the eval/ folder)
file_path = "../exp_output/nova3r_points_cond_finetune_complete/nova3r_training/version_10/test_scores.csv"

# Load the data
df = pd.read_csv(file_path)
display(df.head())

,CD_16384_full,f_score_0.1_full,f_score_0.05_full,f_score_0.02_full,seq_name,id
0,"(0.017021529376506805, 1)","(0.9955477714538574, 1)","(0.9773210287094116, 1)","(0.741004467010498, 1)",replica_pano_large_apartment_2_003,0
1,"(0.016455871984362602, 1)","(0.9958614110946655, 1)","(0.9785892367362976, 1)","(0.7671368718147278, 1)",replica_pano_large_apartment_2_003,1
2,"(0.017519770190119743, 1)","(0.9943444728851318, 1)","(0.9726645350456238, 1)","(0.7339989542961121, 1)",replica_pano_large_apartment_2_003,2
3,"(0.01672236993908882, 1)","(0.9963745474815369, 1)","(0.9802986979484558, 1)","(0.7517696619033813, 1)",replica_pano_large_apartment_2_003,3
4,"(0.017736051231622696, 1)","(0.9947500824928284, 1)","(0.9725487232208252, 1)","(0.7211207747459412, 1)",replica_pano_large_apartment_2_003,4


In [16]:
# Identify the metric columns (everything except seq_name and id)
metric_cols = [col for col in df.columns if col not in ['seq_name', 'id']]

def parse_metric(val_str):
    """Parses a string like '(0.07, 1)' into a Python tuple."""
    if pd.isna(val_str):
        return 0.0, 0
    if isinstance(val_str, str):
        try:
            return ast.literal_eval(val_str)
        except Exception:
            return 0.0, 0
    return 0.0, 0

def get_weighted_average(series):
    """Calculates the weighted average for a pandas Series containing tuple strings."""
    parsed = series.apply(parse_metric)
    scores = parsed.apply(lambda x: x[0])
    weights = parsed.apply(lambda x: x[1])
    
    total_weight = weights.sum()
    if total_weight > 0:
        return (scores * weights).sum() / total_weight
    return 0.0

## Total Metrics
Calculations for the total combined weighted averages across all sequences.

In [17]:
print("--- Total Weighted Averages ---")
total_metrics = {}
for col in metric_cols:
    total_metrics[col] = get_weighted_average(df[col])
    print(f"{col}: {total_metrics[col]:.6f}")

--- Total Weighted Averages ---
CD_16384_full: 0.020106
f_score_0.1_full: 0.983827
f_score_0.05_full: 0.954563
f_score_0.02_full: 0.707010


## Per-Scene Metrics
Calculations for average metrics individually grouped by `seq_name`.

In [18]:

scene_results = []

for scene_name, group in df.groupby('seq_name'):
    row = {'seq_name': scene_name}
    # Count the number of total samples/items per scene
    row['num_samples'] = len(group)  
    for col in metric_cols:
        row[col] = get_weighted_average(group[col])
    scene_results.append(row)

scene_df = pd.DataFrame(scene_results)
display(scene_df)

,seq_name,num_samples,CD_16384_full,f_score_0.1_full,f_score_0.05_full,f_score_0.02_full
0,replica_pano_hotel_0_000,10,0.014917,0.996318,0.979443,0.810729
1,replica_pano_large_apartment_0_004,11,0.027873,0.943725,0.902818,0.707439
2,replica_pano_large_apartment_2_001,9,0.022998,0.994023,0.954185,0.504691
3,replica_pano_large_apartment_2_002,10,0.017132,0.994815,0.965738,0.750996
4,replica_pano_large_apartment_2_003,10,0.017123,0.995286,0.975767,0.740921


In [21]:
scene_results = []

for scene_name, group in df.groupby('seq_name'):
    row = {'seq_name': scene_name}
    # Count the number of total samples/items per scene
    row['num_samples'] = len(group)  
    for col in metric_cols:
        row[col] = get_weighted_average(group[col])
    scene_results.append(row)

scene_df = pd.DataFrame(scene_results)
display(scene_df)

,seq_name,num_samples,CD_16384_full,f_score_0.1_full,f_score_0.05_full,f_score_0.02_full
0,replica_pano_hotel_0_000,100,0.022301,0.987643,0.936103,0.590258
1,replica_pano_large_apartment_0_004,100,0.018476,0.993456,0.963895,0.718643
2,replica_pano_large_apartment_2_001,100,0.022593,0.992677,0.951061,0.535655
3,replica_pano_large_apartment_2_002,100,0.018124,0.993808,0.963665,0.722609
4,replica_pano_large_apartment_2_003,100,0.017297,0.995255,0.975858,0.733721
